## Part 1：语音交互

### 0. 基础环境安装与准备

在开始之前，我们需要配置好相关的依赖库。
请在你的本地 Miniconda 环境中运行以下命令：

```bash
# 安装基础通信库
pip install setuptools requests websocket-client

# 安装多媒体相关库（用于摄像头和麦克风）
pip install opencv-python SpeechRecognition

# 安装 PyAudio
pip install pyaudio

# 安装 Playsound
pip install playsound==1.2.2

```

接下来，安装华为云语音交互专属 SDK（定制版）：
- 下载 SDK 压缩包：[点击此处下载](https://sis-sdk-repository.obs.cn-north-1.myhuaweicloud.com/python/huaweicloud-python-sdk-sis-1.8.6.tar.gz)
- 解压后，在终端进入 huaweicloud-python-sdk-sis-1.8.6 文件夹
- 执行安装命令：`python setup.py install --user`


#### 播放声音

In [1]:
from playsound import playsound

audio_path = '16k16bit.wav'
playsound(audio_path)

KeyboardInterrupt: 

In [ ]:
# Mac备用
from IPython.display import Audio, display
display(Audio('16k16bit.wav', autoplay=True))

#### 麦克风录音

In [13]:
import speech_recognition as sr

r = sr.Recognizer() # 使用SpeechRecognition库，自动判断音频设备接口
mic = sr.Microphone()

with mic as source:
    # 提前适应环境噪声，只需执行一次，不要放在 while 循环里，否则每次录音前都会卡顿 1 秒
    r.adjust_for_ambient_noise(source)

try:
    print("\n👂 Listening... 请说话，停顿将自动结束录音") 
    with mic as source:
        # 开始监听，监听到停顿时自动结束
        audio = r.listen(source) 
        
    # 转换为 16000 采样率，convert_width=2 确保是 16-bit (2 bytes) 深度
    audio_data = audio.get_wav_data(convert_rate=16000, convert_width=2)
    
    save_path = "temp_command.wav"
    with open(save_path, "wb") as f:
        f.write(audio_data)
    print(f"💾 录音已保存至: {save_path}")

except Exception as e:
    print(f"\n❌ 发生错误: {e}")



👂 Listening... 请说话，停顿将自动结束录音
💾 录音已保存至: temp_command.wav


### 1. 语音识别（一句话）

[录音文件识别_Python SDK_SDK参考_语音交互服务 SIS-华为云](https://support.huaweicloud.com/sdkreference-sis/sis_05_0052.html)

<div class="alert alert-warning">
<b>⚠️ 警告：</b> 访问密钥（AK/SK）相当于您的云端密码，请务必妥善保管，切勿上传至 GitHub 等公开平台！
</div>

In [3]:
from huaweicloud_sis.client.asr_client import AsrCustomizationClient
from huaweicloud_sis.bean.asr_request import AsrCustomShortRequest
from huaweicloud_sis.exception.exceptions import ClientException
from huaweicloud_sis.exception.exceptions import ServerException
from huaweicloud_sis.utils import io_utils
from huaweicloud_sis.bean.sis_config import SisConfig
import json

# 鉴权参数 (请替换为你自己的凭证，切勿泄露)
ak = 'HPUAECYCEZPEWS2YIMZ1'
sk = 'LhWVRfAAZLSdCdEm6A7fiuwUg4X77To418OWsD9l'
region = 'cn-east-3'
project_id = '5860acea06914e2ab9351d92286c4677'

"""
    请正确填写音频格式和模型属性字符串
    1. 音频格式一定要相匹配.
         例如wav音频，格式是wav。具体参考api文档。
         例如音频是pcm格式，并且采样率为8k，则格式填写pcm8k16bit。
         如果返回audio_format is invalid 说明该文件格式不支持。具体支持哪些音频格式，需要参考一些api文档。

    2. 音频采样率要与属性字符串的采样率要匹配。
         例如wav本身是16k采样率，属性选择chinese_8k_common, 会返回'audio_format' is not match model
"""

# 一句话识别参数，以音频文件的base64编码传入，1min以内音频
path = './16k16bit.wav'                 # 文件位置, 需要具体到文件，如D:/test.wav
path_audio_format = 'wav'               # 音频格式，如wav等，详见api文档
path_property = 'chinese_16k_general'   # language_sampleRate_domain, 如chinese_8k_common，详见api文档


def sasr_example():
    """ 一句话识别示例 """
    # step1 初始化客户端
    config = SisConfig()
    config.set_connect_timeout(10)  # 设置连接超时
    config.set_read_timeout(10)  # 设置读取超时
    asr_client = AsrCustomizationClient(ak, sk, region, project_id, sis_config=config)

    # step2 构造请求
    data = io_utils.encode_file(path)
    asr_request = AsrCustomShortRequest(path_audio_format, path_property, data)
    # 所有参数均可不设置，使用默认值
    # 设置是否添加标点，yes or no，默认no
    asr_request.set_add_punc('yes')
    # 设置是否将语音中数字转写为阿拉伯数字，yes or no，默认yes
    asr_request.set_digit_norm('yes')
    # 设置是否添加热词表id，没有则不填
    # asr_request.set_vocabulary_id(None)
    # 设置是否需要word_info，yes or no, 默认no
    asr_request.set_need_word_info('no')

    # step3 发送请求，返回结果,返回结果为json格式
    result = asr_client.get_short_response(asr_request)
    print(json.dumps(result, indent=2, ensure_ascii=False))


if __name__ == '__main__':
    try:
        sasr_example()
    except ClientException as e:
        print(e)
    except ServerException as e:
        print(e)

{
  "trace_id": "d20579e0-566d-437c-bda3-ba52e2ca5ae2",
  "result": {
    "text": "华为致力于把数字世界带入每个人，每个家庭，每个组织，构建万物互联的智能世界。",
    "score": 0.8920642137527466
  }
}


### 2. 语音合成（一句话）

[语音合成_Python SDK_SDK参考_语音交互服务 SIS-华为云](https://support.huaweicloud.com/sdkreference-sis/sis_05_0055.html)

In [4]:
from huaweicloud_sis.client.tts_client import TtsCustomizationClient
from huaweicloud_sis.bean.tts_request import TtsCustomRequest
from huaweicloud_sis.bean.sis_config import SisConfig
from huaweicloud_sis.exception.exceptions import ClientException
from huaweicloud_sis.exception.exceptions import ServerException
import json

# 鉴权参数 (请替换为你自己的凭证，切勿泄露)
ak = 'HPUAECYCEZPEWS2YIMZ1'
sk = 'LhWVRfAAZLSdCdEm6A7fiuwUg4X77To418OWsD9l'
region = 'cn-east-3'
project_id = '5860acea06914e2ab9351d92286c4677'

def ttsc_example():
    """ 语音合成demo """
    text = '上海交通大学学生创新中心欢迎您！'           # 待合成文本，不超过500字
    path = './test.wav'                              # 保存路径，如D:/test.wav。 可在设置中选择不保存本地

    # step1 初始化客户端
    config = SisConfig()
    config.set_connect_timeout(10)       # 设置连接超时，单位s
    config.set_read_timeout(10)          # 设置读取超时，单位s
    ttsc_client = TtsCustomizationClient(ak, sk, region, project_id, sis_config=config)

    # step2 构造请求
    ttsc_request = TtsCustomRequest(text)
    # 设置请求，所有参数均可不设置，使用默认参数
    # 设置属性字符串， language_speaker_domain, 默认chinese_xiaoyan_common, 参考api文档
    ttsc_request.set_property('chinese_xiaoyu_common')
    # 设置音频格式，默认wav，可选mp3和pcm
    ttsc_request.set_audio_format('wav')
    # 设置采样率，8000 or 16000, 默认8000
    ttsc_request.set_sample_rate('8000')
    # 设置音量，[0, 100]，默认50
    ttsc_request.set_volume(50)
    # 设置音高, [-500, 500], 默认0
    ttsc_request.set_pitch(0)
    # 设置音速, [-500, 500], 默认0
    ttsc_request.set_speed(0)
    # 设置是否保存，默认False
    ttsc_request.set_saved(True)
    # 设置保存路径，只有设置保存，此参数才生效
    ttsc_request.set_saved_path(path)

    # step3 发送请求，返回结果。如果设置保存，可在指定路径里查看保存的音频。
    result = ttsc_client.get_ttsc_response(ttsc_request)
    print(json.dumps(result, indent=2, ensure_ascii=False))


if __name__ == '__main__':
    try:
        ttsc_example()
    except ClientException as e:
        print(e)
    except ServerException as e:
        print(e)

{
  "result": {
    "data": "..."
  },
  "trace_id": "d42bdea8-791e-43b4-99a6-869e6667f21d",
  "is_saved": true,
  "saved_path": "./test.wav"
}


### 3. 语音克隆：注册和查询音色

[声音复刻接口_Python SDK_SDK参考_语音交互服务 SIS-华为云](https://support.huaweicloud.com/sdkreference-sis/sis_05_0103.html)

[声音注册查询_Python SDK_SDK参考_语音交互服务 SIS-华为云](https://support.huaweicloud.com/sdkreference-sis/sis_05_0102.html)

In [15]:
import base64
import os
from huaweicloud_sis.bean.vcs_register_voice_request import RegisterVoiceRequest
from huaweicloud_sis.client.vcs_client import VcsClient
from huaweicloud_sis.bean.sis_config import SisConfig

# 鉴权参数 (请替换为你自己的凭证，切勿泄露)
ak = 'HPUAECYCEZPEWS2YIMZ1'
sk = 'LhWVRfAAZLSdCdEm6A7fiuwUg4X77To418OWsD9l'
region = 'cn-east-3'
project_id = '5860acea06914e2ab9351d92286c4677'

path = "./temp_command.wav"        # 参考语音路径，要求采样率不小于16k，例如 r"D://test.wav"，参考语音时间不短于5秒
voice_name = "abc"                 # 音色名称，可以使用注册音色或预置音色
service_endpoint = "https://sis-ext.cn-east-3.myhuaweicloud.com" # 可选，自定义endpoint

def register_voice_example(path, voice_name):
    with open(path, 'rb') as f:
        data = f.read()
        data = str(base64.b64encode(data), 'utf-8')
    config = SisConfig()
    config.set_connect_timeout(10)  # 设置连接超时，单位s
    config.set_read_timeout(60)  # 设置读取超时，单位s
    vcs_client = VcsClient(ak, sk, region, project_id, sis_config=config, service_endpoint=service_endpoint)
    register_voice_request = RegisterVoiceRequest(data, voice_name)
    result = vcs_client.register_voice(register_voice_request)
    print(result)

def query_example():
    config = SisConfig()
    config.set_connect_timeout(10)  # 设置连接超时，单位s
    config.set_read_timeout(60)  # 设置读取超时，单位s
    vcs_client = VcsClient(ak, sk, region, project_id, sis_config=config, service_endpoint=service_endpoint)
    result = vcs_client.query_voice_name(50, 0) # 默认查询10条，支持分页查询
    print(result)

if __name__ == '__main__':
    # 两个接口可单独运行
    # register_voice_example(path, voice_name) # 注册
    query_example() # 查询

{'result': {'voices': [{'voice_name': 'bonder', 'language': 'chinese'}, {'voice_name': 'ch', 'language': 'chinese'}, {'voice_name': 'cys', 'language': 'chinese'}, {'voice_name': 'deersnow', 'language': 'chinese'}, {'voice_name': 'demovoice', 'language': 'chinese'}, {'voice_name': 'DMH', 'language': 'chinese'}, {'voice_name': 'fsy', 'language': 'chinese'}, {'voice_name': 'girl1', 'language': 'chinese'}, {'voice_name': 'hhhhhhh', 'language': 'chinese'}, {'voice_name': 'hj', 'language': 'chinese'}, {'voice_name': 'HLF', 'language': 'chinese'}, {'voice_name': 'hzy', 'language': 'chinese'}, {'voice_name': 'josie', 'language': 'chinese'}, {'voice_name': 'jyd', 'language': 'chinese'}, {'voice_name': 'Liyuu_normal', 'language': 'chinese'}, {'voice_name': 'lmy', 'language': 'chinese'}, {'voice_name': 'lr', 'language': 'chinese'}, {'voice_name': 'lr1', 'language': 'chinese'}, {'voice_name': 'LYX', 'language': 'chinese'}, {'voice_name': 'M', 'language': 'chinese'}, {'voice_name': 'MINKEONG', 'lan

### 4. 语音克隆：使用已有音色进行合成

使用该脚本前，保证要使用的声音已经注册。若没有注册声音，请使用上一步骤代码，注册声音和查询声音。

In [16]:
from huaweicloud_sis.client.vcs_stream_client import VcsStreamClient
from huaweicloud_sis.bean.vcs_stream_request import VcsStreamRequest
from huaweicloud_sis.bean.callback import VcsStreamCallBack
from huaweicloud_sis.bean.sis_config import SisConfig
import os

# 鉴权参数 (请替换为你自己的凭证，切勿泄露)
ak = 'HPUAECYCEZPEWS2YIMZ1'
sk = 'LhWVRfAAZLSdCdEm6A7fiuwUg4X77To418OWsD9l'
region = 'cn-east-3'
project_id = '5860acea06914e2ab9351d92286c4677'
service_endpoint = "https://sis-ext.cn-east-3.myhuaweicloud.com" # 可选，自定义endpoint

text_path = ""  # 可选 待合成文本存储路径,例如"D://text.txt",要求文本中，每行句子不超过300字符
audio_path = r"clone.wav"       # 待合成的音频保存路径，如test.wav
voice_name = 'abc'     # 音色名称，可以使用注册音色或预置音色

class MyCallback(VcsStreamCallBack):
    """ 回调类，用户需要在对应方法中实现自己的逻辑，其中on_response必须重写 """

    def __init__(self, save_path):
        self._f = open(save_path, 'wb')

    def on_open(self):
        """ websocket连接成功会回调此函数 """
        print('websocket connect success')

    def on_start(self, message):
        """
            websocket 开始识别回调此函数
        :param message: 传入信息
        :return: -
        """
        print('webscoket start to recognize, %s' % message)

    def on_response(self, data):
        """
            回调返回的音频合成数据，byte数组格式
        :param data byte数组，合成的音频数据
        :return: -
        """
        print('receive data %d' % len(data))
        self._f.write(data)

    def on_end(self, message):
        """
            websocket 结束识别回调此函数
        :param message: 传入信息
        :return: -
        """
        print('websocket is ended, %s' % message)
        self._f.close()

    def on_close(self):
        """ websocket关闭会回调此函数 """
        print('websocket is closed')
        self._f.close()

    def on_error(self, error):
        """
            websocket出错回调此函数
        :param error: 错误信息
        :return: -
        """
        print('websocket meets error, the error is %s' % error)
        self._f.close()


def vcs_stream_example(audio_path, voice_name):
    """
        声音复刻流式demo
    """
    # step1 初始化VcsStreamClient
    my_callback = MyCallback(audio_path)
    config = SisConfig()
    # 设置连接超时,默认是10
    config.set_connect_timeout(10)
    # 设置读取超时, 默认是10
    config.set_read_timeout(10)
    # 设置websocket等待时间
    config.set_websocket_wait_time(20)
    # websocket暂时不支持使用代理
    vcs_stream_client = VcsStreamClient(ak=ak, sk=sk, use_aksk=True, region=region, project_id=project_id, callback=my_callback,config=config)

    # step2 构造请求
    vcs_stream_request = VcsStreamRequest()
    # 设置音色字符串，可以使用注册音色或预置音色
    vcs_stream_request.set_voice_name(voice_name)
    # 设置音频格式, pcm or mp3, 可参考api文档
    vcs_stream_request.set_audio_format('mp3')
    # 设置采样率，8000 or 16000 or 24000, 默认16000
    vcs_stream_request.set_sample_rate('16000')
    # 设置音量，[0, 100]，默认50
    vcs_stream_request.set_volume(50)
    # 设置音高, [-500, 500], 默认0
    vcs_stream_request.set_pitch(0)
    # 设置音速, [-500, 500], 默认0
    vcs_stream_request.set_speed(0)
    # ONLY模式下，用户发送一次文本，服务端流式返回语音数据；MULTI模式下，支持用户多次发送文本，服务端流式返回语音数据，可用于大模型输出实时播报等场景。
    text_pieces = 'ONLY' #'MULTI'
    vcs_stream_request.set_text_pieces(text_pieces)
    if text_pieces == 'MULTI':

        # 发送开始请求、发送文本、发送end请求
        vcs_stream_client.sendStart(vcs_stream_request)

        # 发送文本，可连续发送多次，合成结果可以通过监听器获取
        # 方法一
        vcs_stream_client.sendMsg("你好")
        vcs_stream_client.sendMsg("欢迎使用华为云语音交互服务。")

        # 方法二
        # vcs_stream_client.send_msg_batch(text_path)  # 传入待合成文本储存路径

        # 文本发送完毕，发送结束指令
        vcs_stream_client.sendEnd()
    else:
        # ONLY模式下，发送一次文本
        vcs_stream_request.set_text("这里是上海交通大学学生创新中心，欢迎您的来访。")
        vcs_stream_client.synthesis(vcs_stream_request)

if __name__ == '__main__':
    vcs_stream_example(audio_path, voice_name)

[2026-03-30 13:50:26,122] - [INFO] - [Websocket connected]
[2026-03-30 13:50:26,129] - [INFO] - [vcs stream websocket open]


websocket connect success


[2026-03-30 13:50:26,327] - [ERROR] - [receive error resp {"resp_type":"ERROR","trace_id":"4c954dec-189d-4865-83bc-38ecb19cd6cb","error_code":"SIS.1203","error_msg":"'voice_name' invalid, not exist"}]


websocket meets error, the error is {"resp_type":"ERROR","trace_id":"4c954dec-189d-4865-83bc-38ecb19cd6cb","error_code":"SIS.1203","error_msg":"'voice_name' invalid, not exist"}


[2026-03-30 13:50:36,466] - [INFO] - [vcs stream websocket close]


websocket is closed


## Part 2: 文字识别（OCR）

### 1. 调用本地摄像头拍摄图片

`pip install opencv-python`

现在来测试一下你的电脑摄像头。我们将使用 OpenCV 库拍摄一张照片并保存到本地。

> ⚠️ **注意**：运行此代码时，如果系统弹窗请求摄像头权限，请点击“允许”。

In [ ]:
import cv2

print("Take picture")
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
if ret:
    cv2.imwrite("capture.jpg", frame)
else:
    print("no picture")
cap.release()
cv2.destroyAllWindows()

### 2. 读取本地文件进行OCR识别

[Python SDK_SDK参考_文字识别 OCR-华为云](https://support.huaweicloud.com/sdkreference-ocr/ocr_04_0006.html)

`pip install huaweicloudsdkcore huaweicloudsdkocr`

In [ ]:
import os
import base64
from huaweicloudsdkcore.auth.credentials import BasicCredentials
from huaweicloudsdkocr.v1.region.ocr_region import OcrRegion
from huaweicloudsdkcore.exceptions import exceptions
from huaweicloudsdkocr.v1 import *

if __name__ == "__main__":
    ak = ''
    sk = ''

    credentials = BasicCredentials(ak, sk)

    client = OcrClient.new_builder() \
        .with_credentials(credentials) \
        .with_region(OcrRegion.value_of("cn-east-3")) \
        .build()

    try:
        # 1. 读取图片文件并转换为 Base64
        image_path = "./capture.jpg"
        with open(image_path, "rb") as f:
            image_base64 = base64.b64encode(f.read()).decode("utf-8")

        # 2. 构建请求体
        request = RecognizeGeneralTextRequest()
        request.body = GeneralTextRequestBody(
            image=image_base64   # 传递 Base64 字符串
        )

        # 3. 发送请求
        response = client.recognize_general_text(request)
        print(response)
    except exceptions.ClientRequestException as e:
        print(e.status_code)
        print(e.request_id)
        print(e.error_code)
        print(e.error_msg)


## Part 3：综合任务

**🎯 任务目标**：开发一个持续监听的智能小助手

**📝 具体流程**：
1. 程序持续运行，监听你的麦克风语音指令。
2. 当你对着麦克风说出 **“拍照”** 时，机器人说出：“我要开始了”，程序调用摄像头拍摄一张照片（保存为 `capture.jpg`）。
3. 接着，调用华为云 OCR 接口识别照片里的文字。
4. 调用华为云 TTS 接口，将识别到的文字转换成语音并播放出来。
5. 当你说出 **“退出”** 时，结束程序运行。

In [1]:
### Complete your code here...
import os
import json
import base64
import cv2
import playsound
import speech_recognition as sr

from huaweicloud_sis.client.asr_client import AsrCustomizationClient
from huaweicloud_sis.bean.asr_request import AsrCustomShortRequest
from huaweicloud_sis.bean.sis_config import SisConfig
from huaweicloud_sis.client.tts_client import TtsCustomizationClient
from huaweicloud_sis.bean.tts_request import TtsCustomRequest
from huaweicloud_sis.exception.exceptions import ClientException, ServerException
from huaweicloud_sis.utils import io_utils

from huaweicloudsdkcore.auth.credentials import BasicCredentials
from huaweicloudsdkcore.exceptions import exceptions
from huaweicloudsdkocr.v1.region.ocr_region import OcrRegion
from huaweicloudsdkocr.v1 import *


# =========================
# 1. 填写你自己的华为云信息
# =========================
ak = 'HPUAECYCEZPEWS2YIMZ1'
sk = 'LhWVRfAAZLSdCdEm6A7fiuwUg4X77To418OWsD9l'
region = 'cn-east-3'
project_id = '5860acea06914e2ab9351d92286c4677'

# =========================
# 2. 初始化麦克风
# =========================
r = sr.Recognizer()
mic = sr.Microphone()

with mic as source:
    print("正在进行环境噪声校准，请稍等...")
    r.adjust_for_ambient_noise(source, duration=1)
print("环境噪声校准完成。")


# =========================
# 3. 语音识别：录音并保存
# =========================
def record_audio(save_path="temp_command.wav"):
    try:
        print("\n👂 Listening... 请说话，停顿将自动结束录音")
        with mic as source:
            audio = r.listen(source, timeout=5, phrase_time_limit=4)

        audio_data = audio.get_wav_data(convert_rate=16000, convert_width=2)
        with open(save_path, "wb") as f:
            f.write(audio_data)

        print(f"💾 录音已保存至: {save_path}")
        return save_path
    except Exception as e:
        print("录音失败：", e)
        return None


# =========================
# 4. ASR：音频转文字
# =========================
def speech_to_text(audio_path):
    try:
        config = SisConfig()
        config.set_connect_timeout(10)
        config.set_read_timeout(10)

        asr_client = AsrCustomizationClient(
            ak, sk, region, project_id, sis_config=config
        )

        data = io_utils.encode_file(audio_path)
        asr_request = AsrCustomShortRequest('wav', 'chinese_16k_general', data)
        asr_request.set_add_punc('yes')
        asr_request.set_digit_norm('yes')
        asr_request.set_need_word_info('no')

        result = asr_client.get_short_response(asr_request)
        print("ASR原始结果：")
        print(json.dumps(result, indent=2, ensure_ascii=False))

        # 尽量兼容不同返回格式
        text = ""
        if isinstance(result, dict):
            if "result" in result:
                if isinstance(result["result"], dict):
                    text = result["result"].get("text", "")
                elif isinstance(result["result"], str):
                    text = result["result"]
            if not text:
                text = result.get("text", "")

        return text.strip()

    except (ClientException, ServerException) as e:
        print("语音识别失败：", e)
        return ""


# =========================
# 5. 拍照
# =========================
def take_picture(image_path="capture.jpg"):
    print("📷 正在拍照...")
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("无法打开摄像头")
        return None

    ret, frame = cap.read()
    cap.release()

    if ret:
        cv2.imwrite(image_path, frame)
        print(f"图片已保存到: {image_path}")
        return image_path
    else:
        print("拍照失败")
        return None


# =========================
# 6. OCR：图片转文字
# =========================
def ocr_recognize(image_path):
    try:
        credentials = BasicCredentials(ak, sk)

        client = OcrClient.new_builder() \
            .with_credentials(credentials) \
            .with_region(OcrRegion.value_of(region)) \
            .build()

        with open(image_path, "rb") as f:
            image_base64 = base64.b64encode(f.read()).decode("utf-8")

        request = RecognizeGeneralTextRequest()
        request.body = GeneralTextRequestBody(image=image_base64)

        response = client.recognize_general_text(request)
        print("OCR原始结果：")
        print(response)

        # 尝试从返回结果里提取文字
        text_result = ""

        if hasattr(response, "result") and response.result:
            result_obj = response.result

            # 常见情况：words_block_list
            if hasattr(result_obj, "words_block_list") and result_obj.words_block_list:
                texts = []
                for item in result_obj.words_block_list:
                    if hasattr(item, "words") and item.words:
                        texts.append(item.words)
                text_result = "\n".join(texts)

            # 兜底：如果还有 text / words 等字段
            if not text_result:
                if hasattr(result_obj, "text") and result_obj.text:
                    text_result = result_obj.text
                elif hasattr(result_obj, "words") and result_obj.words:
                    text_result = result_obj.words

        return text_result.strip()

    except exceptions.ClientRequestException as e:
        print("OCR请求失败：")
        print(e.status_code)
        print(e.request_id)
        print(e.error_code)
        print(e.error_msg)
        return ""
    except Exception as e:
        print("OCR识别失败：", e)
        return ""


# =========================
# 7. TTS：文字转语音并播放
# =========================
import time

def text_to_speech(text, save_path=None):
    try:
        config = SisConfig()
        config.set_connect_timeout(10)
        config.set_read_timeout(10)

        tts_client = TtsCustomizationClient(
            ak, sk, region, project_id, sis_config=config
        )

        if save_path is None:
            save_path = f"tts_{int(time.time() * 1000)}.wav"

        tts_request = TtsCustomRequest(text)
        tts_request.set_property('chinese_xiaoyu_common')
        tts_request.set_audio_format('wav')
        tts_request.set_sample_rate('8000')
        tts_request.set_volume(50)
        tts_request.set_pitch(0)
        tts_request.set_speed(0)
        tts_request.set_saved(True)
        tts_request.set_saved_path(save_path)

        result = tts_client.get_ttsc_response(tts_request)
        print("TTS结果：")
        print(json.dumps(result, indent=2, ensure_ascii=False))

        if os.path.exists(save_path):
            playsound.playsound(save_path)
            return True
        return False

    except (ClientException, ServerException) as e:
        print("语音合成失败：", e)
        return False
    except Exception as e:
        print("播放失败：", e)
        return False

# =========================
# 8. 主循环
# =========================
print("\n===== 语音 + OCR 综合任务开始 =====")
print("请说“拍照”来拍摄并识别文字，说“退出”结束程序。")

while True:
    audio_path = record_audio("temp_command.wav")
    if not audio_path:
        continue

    command = speech_to_text(audio_path)
    print("识别到的指令：", command)

    # 去掉标点，增强鲁棒性
    command_simple = command.replace("。", "").replace("，", "").replace("！", "").replace(" ", "")

    if "退出" in command_simple:
        print("程序结束。")
        text_to_speech("好的，程序结束。")
        break

    elif "拍照" in command_simple:
        print("识别到拍照指令，准备开始。")
        text_to_speech("我要开始了")
        time.sleep(1)

        image_path = take_picture("capture.jpg")
        if image_path is None:
            text_to_speech("拍照失败，请检查摄像头。")
            continue

        ocr_text = ocr_recognize(image_path)
        print("OCR识别结果：")
        print(ocr_text)

        if ocr_text:
            speak_text = "识别到的文字是：" + ocr_text
        else:
            speak_text = "没有识别到文字。"

        text_to_speech(speak_text)

    else:
        print("未识别到有效指令，请再说一次。")
        text_to_speech("没有听清，请说拍照或者退出。")

正在进行环境噪声校准，请稍等...
环境噪声校准完成。

===== 语音 + OCR 综合任务开始 =====
请说“拍照”来拍摄并识别文字，说“退出”结束程序。

👂 Listening... 请说话，停顿将自动结束录音
💾 录音已保存至: temp_command.wav
ASR原始结果：
{
  "trace_id": "200f47b9-b919-4487-bc92-d006ff34468b",
  "result": {
    "text": "拍照。",
    "score": 0.8608354926109314
  }
}
识别到的指令： 拍照。
识别到拍照指令，准备开始。
TTS结果：
{
  "result": {
    "data": "..."
  },
  "trace_id": "68d31ff6-62fa-42bf-9198-bed671fb39d7",
  "is_saved": true,
  "saved_path": "tts_1774854806993.wav"
}
📷 正在拍照...
图片已保存到: capture.jpg
OCR原始结果：
{"result": {"direction": -1.0, "words_block_count": 0, "words_block_list": []}, "X-Request-Id": "30234f4e88298a6245d58e0e2a020ae0"}
OCR识别结果：

TTS结果：
{
  "result": {
    "data": "..."
  },
  "trace_id": "0516ee48-a4a4-416d-bdf7-d4ddefa8a835",
  "is_saved": true,
  "saved_path": "tts_1774854812421.wav"
}

👂 Listening... 请说话，停顿将自动结束录音
💾 录音已保存至: temp_command.wav
ASR原始结果：
{
  "trace_id": "948b7ada-2125-470b-83be-7bd3e768e451",
  "result": {
    "text": "我发现我靠没开声音，他听不见。",
    "score": 0.62076181173